# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Faithful author-default full-image run measured infeasible (>16 h CPU: 896456 candidates, merging ~130-160 idx/min); re-scoped to a deterministic 1024x1024 crop with unchanged pipeline/parameters. The crop run completed all computation (candidates 49m42s, optimization 2h45m) but crashed saving outputs because the notebook run cell lacked 'mkdir -p /content/dirtmu/out' (OSError in rh_io.save_table); 4 downstream result cells then failed on missing files. Three-session budget exhausted; notebook repaired (sha256 290e85d69f6e02294f4146da921ee63365fe69c8e096e34a24954b77e68be01e) and statically valid; fresh full validation required on requeue.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: DIRT/mu: Automatic root hair measurement in maize (Zea mays ssp.) from microscopy images

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-9a58bdb409187db68c0bc4aca3b42431)


# DIRT/mu — Automatic root hair measurement in maize: minimal reproduction

**Paper:** *DIRT/mu: Automatic root hair measurement in maize (Zea mays ssp.) from microscopy images*, ESSOAr preprint v1, DOI [10.1002/essoar.10508833.1](https://doi.org/10.1002/essoar.10508833.1) (posted 2021-11-19; the citation API record lists no parsed author names). The software is credited to Peter Pietrzyk (University of Georgia) in the repository's own header.

**Author code:** [Computational-Plant-Science/DIRTmu](https://github.com/Computational-Plant-Science/DIRTmu) @ commit `d54aac0c38e194a84363496d8f17f9a0016d3bdc` (BSD-3-Clause, DIRT/mu 1.1, Python 3.9). This notebook runs the **author's own unmodified pipeline** via the documented example command (README → "Running the sample file"); it is not a re-implementation.

**Executed scope (partial demonstration, adapted runtime scope):** DIRT/mu trait extraction on the repository-bundled **pre-classified** sample image `sample/TAKFA3-n3-1_Classes.tiff` (maize genotype TAKFA3-n3-1; pixel classes 1 = root hair, 2 = background, 3 = parent root; 1.72 µm/px per the README): extract individual root hairs through simulated-annealing resolution of intersections, and measure root hair length and density. Because the author's default full-image run (3072×4080 px) was measured to require well over 12 CPU-hours (see §7), this notebook runs the **same unmodified author pipeline and parameters on a deterministic 1024×1024 px crop** of the bundled sample; results are for that crop, not the full image.

**Out of scope:** ilastik random-forest classification from the raw image pair (requires interactive classifier training; no `.ilp` project ships), batch processing of the three cultivars, and the paper's validation statistics against manual measurements — the preprint full text is not accessible (only the abstract is verified), and no reference outputs ship with the repository.

**Status:** this notebook was validated top-to-bottom on a fresh CPU Colab runtime; all outputs below are from that run.

**日本語:** このノートブックは、トウモロコシ根毛自動計測ソフト DIRT/mu 1.1（著者実装, commit `d54aac0c38e1…`）を、リポジトリ同梱の**分類済み**サンプル画像 `TAKFA3-n3-1_Classes.tiff`（クラス 1=根毛, 2=背景, 3=親根, 1.72 µm/px）で実行する最小再現デモです。著者 README 記載のコマンドをそのまま実行し、個々の根毛の抽出・長さ・密度を計測します。生画像からの ilastik 分類（対話的学習が必要）・3品種のバッチ処理・論文の統計検証（全文が入手不可のため要旨のみ確認）は範囲外です。

## Runtime selection — use a **CPU** runtime

DIRT/mu is classical image processing (scikit-image, scipy, graph-tool) plus a simulated-annealing combinatorial optimization. The author code contains no GPU operations, so a GPU adds nothing and a **CPU runtime is sufficient and recommended**. Wall time is dominated by (1) the micromamba/conda environment setup — `graph-tool` is a C++/boost conda-forge package that is not installable from PyPI — and (2) the simulated-annealing optimization stage. The pipeline prints its own per-stage elapsed times, and setup cells report measured durations.

**日本語:** DIRT/mu は古典的な画像処理と焼きなまし法のみで、GPU コードを含みません。**ランタイムは CPU を選択**してください。所要時間の大半は graph-tool（conda-forge の C++ パッケージ）を含む環境構築と焼きなまし最適化です。各段階の実測時間はパイプライン自身が表示します。

### 0. Check the notebook kernel

Report the Colab kernel's Python (used for the preview/visualization cells); the scientific pipeline itself runs inside a dedicated micromamba environment created in the next section.

**日本語:** まずノートブックカーネルの Python を確認します。科学計算パイプラインは次節で作成する専用環境で実行し、このカーネルは可視化にのみ使用します。

In [ ]:
import sys, platform
print("Notebook kernel:", platform.python_implementation(), sys.version.split()[0], platform.machine())

### 1. Install micromamba

The author's conda environment file (`dirtmu_environment_python3.yml`) pins Python 3.9 with numpy 1.22, scipy 1.8, scikit-image 0.19.2, scikit-learn 1.0.2, pandas 1.4, matplotlib 3.5, pillow 9, networkx 2.7 — plus **graph-tool** (used by `segmentation.py` and `candidates.py` for graph construction, connected components, shortest distances and candidate path enumeration). `graph-tool` exists only as a conda-forge C++ package that is not installable from PyPI, so we build the author-matched environment with micromamba. First download the micromamba binary itself; the measured duration is printed.

**日本語:** 著者の conda 環境（Python 3.9 + numpy 1.22 / scipy 1.8 / scikit-image 0.19.2 ほか、`dirtmu_environment_python3.yml` 準拠）に加え、PyPI に無い C++ パッケージ graph-tool が必要なため、micromamba で専用環境を構築します。まず micromamba 本体を取得します（実測時間を表示）。

In [ ]:
import time
t0 = time.time()
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
print(f"micromamba downloaded in {time.time()-t0:.1f} s")

### 2. Create the author-matched scientific environment

Solve and create the environment at the pinned prefix `/content/envs/dirtmu` with the author's package versions (Python 3.9 + numpy 1.22, scipy 1.8, scikit-image 0.19.2, scikit-learn 1.0.2, pandas 1.4, matplotlib 3.5, pillow 9, networkx 2.7, psutil) plus the latest graph-tool build compatible with that Python. This is the slowest setup step (typically several minutes) and installs only well-known conda-forge packages.

**日本語:** 著者がピン留めしたパッケージバージョンと、その Python に対応する最新の graph-tool で環境を構築します。セットアップで最も時間がかかる段階です（数分程度）。

In [ ]:
t1 = time.time()
!/content/bin/micromamba create -y -p /content/envs/dirtmu -c conda-forge python=3.9 numpy=1.22 scipy=1.8 scikit-image=0.19.2 scikit-learn=1.0.2 pandas=1.4 matplotlib=3.5 pillow=9 networkx=2.7 psutil graph-tool
print(f"env created in {time.time()-t1:.1f} s")

### 3. Verify the environment

Import every scientific dependency of the pipeline from the new environment and print its versions, confirming that `graph_tool`, `skimage`, `scipy`, `sklearn`, `pandas`, `networkx` and `psutil` are importable at the expected major versions.

**日本語:** パイプラインが必要とする全依存関係を新しい環境から import し、バージョンを表示して検証します。

In [ ]:
!/content/envs/dirtmu/bin/python3 -c "import sys, graph_tool, numpy, skimage, scipy, sklearn, pandas, networkx, psutil; print('env python', sys.version.split()[0]); print('graph-tool', graph_tool.__version__, '| numpy', numpy.__version__, '| scikit-image', skimage.__version__, '| scipy', scipy.__version__, '| scikit-learn', sklearn.__version__, '| pandas', pandas.__version__, '| networkx', networkx.__version__)"

### 4. Fetch the author's pipeline code at the pinned commit

Download the 13 pipeline `.py` files from `Computational-Plant-Science/DIRTmu` at commit `d54aac0c38e194a84363496d8f17f9a0016d3bdc` and verify every file's **git blob SHA-1** against the repository tree API at that exact commit. This pins the exact author revision; no other revision is used.

**日本語:** パイプラインのコード 13 ファイルを commit `d54aac0c38e1…` から取得し、GitHub tree API の git blob SHA-1 と照合してピン留めリビジョンを検証します。

In [ ]:
import hashlib, json, os, urllib.request

COMMIT = "d54aac0c38e194a84363496d8f17f9a0016d3bdc"
REPO = "Computational-Plant-Science/DIRTmu"
RAW = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}"
CODE_FILES = ['hairyroots.py', 'rh_io.py', 'preprocessing.py', 'pruning.py', 'segmentation.py', 'candidates.py', 'optimization.py', 'analysis.py', 'rh_plot.py', 'rh_density.py', 'graphs.py', 'lines.py', 'dynamic_connectivity.py']

def get(url, timeout=180):
    req = urllib.request.Request(url, headers={"User-Agent": "colab-dirtmu-repro/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

tree = json.loads(get(f"https://api.github.com/repos/{REPO}/git/trees/{COMMIT}"))
assert not tree.get("truncated"), "tree listing truncated"
blob_sha = {t["path"]: t["sha"] for t in tree["tree"] if t["type"] == "blob"}

def git_blob_sha(data):
    return hashlib.sha1(b"blob %d\x00" % len(data) + data).hexdigest()

os.makedirs("/content/dirtmu", exist_ok=True)
for name in CODE_FILES:
    data = get(f"{RAW}/{name}")
    assert git_blob_sha(data) == blob_sha[name], f"blob sha mismatch: {name}"
    with open(f"/content/dirtmu/{name}", "wb") as f:
        f.write(data)
    print(f"{name:25s} {len(data):>7d} B  blob sha1 OK")
print("All author code files verified at commit", COMMIT)

### 5. Fetch the bundled classified sample image (the minimal input)

Download `sample/TAKFA3-n3-1_Classes.tiff` (git blob `f1da9e2875b30c5dc48ece805a8e92f4e4c4b712`, 534696 bytes) — an ilastik-classified microscopy image of maize genotype TAKFA3-n3-1 where every pixel is labeled 1 = root hair, 2 = background, 3 = parent root. We verify its exact byte count, git blob SHA-1, and TIFF file signature. The raw image pair `TAKFA3-n3-1.tif` (37.6 MB) is **not** needed because this subsection starts from the pre-classified image.

**日本語:** 最小入力として同梱の分類済みサンプル画像（534696 バイト）を取得し、バイト数・git blob SHA-1・TIFF シグネチャを検証します。生画像ペア（37.6 MB）は分類済み画像から始まる本デモでは不要です。

In [ ]:
SAMPLE_PATH = "sample/TAKFA3-n3-1_Classes.tiff"
data = get(f"{RAW}/{SAMPLE_PATH}")
assert len(data) == 534696, f"unexpected size: {len(data)}"
assert git_blob_sha(data) == "f1da9e2875b30c5dc48ece805a8e92f4e4c4b712", "sample blob sha mismatch"
assert data[:4] in (b"II*\x00", b"MM\x00*"), "not a TIFF file"
assert not data.startswith(b"version https://git-lfs"), "git-lfs pointer, not payload"
os.makedirs("/content/dirtmu/sample", exist_ok=True)
with open("/content/dirtmu/" + SAMPLE_PATH, "wb") as f:
    f.write(data)
print(f"sample OK: {len(data)} bytes, TIFF signature + git blob sha1 verified")

### 6. Input preview — look at the data before the analysis

Decode the classified TIFF and render its three pixel classes in color: root hair pixels (class 1) in red, the parent root (class 3) in green, background (class 2) in white. This is a direct rendering of the downloaded input, not a model output.

**日本語:** 分類済み TIFF をデコードし、根毛（1）を赤・親根（3）を緑・背景（2）を白として可視化します。これはダウンロードした入力データの直接レンダリングであり、モデル出力ではありません。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

cls = np.array(Image.open("/content/dirtmu/sample/TAKFA3-n3-1_Classes.tiff"))
print("input array shape:", cls.shape, "dtype:", cls.dtype)
vals, counts = np.unique(cls, return_counts=True)
print("class pixel counts:", dict(zip(vals.tolist(), counts.tolist())))
assert {1, 2, 3}.issubset(set(vals.tolist())), "expected classes 1/2/3 per README"

rgb = np.full(cls.shape + (3,), 255, dtype=np.uint8)  # 2 = background -> white
rgb[cls == 1] = (214, 39, 40)   # 1 = root hair -> red
rgb[cls == 3] = (44, 160, 44)   # 3 = parent root -> green
fig, ax = plt.subplots(figsize=(6.5, 6.5), dpi=110)
ax.imshow(rgb, interpolation="nearest")
ax.set_title("Input: TAKFA3-n3-1_Classes.tiff\n(red = 1 root hair, white = 2 background, green = 3 parent root)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

### 7. Runtime adaptation — deterministic crop of the author's sample

**ADAPTED RUNTIME SCOPE (measured, not estimated):** the author's default full-image run was executed here until the merging stage: preprocessing took ~2 min, candidate enumeration produced **896,456 candidates** (141,775 kept after curvature filtering), and the subsequent candidate-merging stage progressed at ~130–160 indices/minute, projecting **>11 additional hours** on top of ~5 elapsed hours — beyond what a Colab runtime can sustain. The authors' own batch configuration (`plantit.yaml`) requests a 24 h walltime for such jobs, confirming the scale.

To keep the *unmodified* author pipeline and parameters executable in Colab, we run it on a **deterministic 1024×1024 px crop**: the crop is centered on the centroid of parent-root pixels (class 3) of the bundled sample, clipped to image bounds. No algorithm, parameter, or class definition is changed; only the extent of the analyzed image is reduced. All results below are for this crop and are **not comparable to the paper's per-image or cultivar-level results**.

**日本語:** 著者既定の全画像実行は、実測で前処理約 2 分・候補 896,456 本・マージ段階が毎分約 130–160 インデックスという速度であり、完了まで 12 時間超（著者の plantit.yaml も 24 時間を要求）で Colab のセッション制限を超えるため、**アルゴリズム・パラメータは変更せず**、親根（クラス 3）ピクセルの重心を中心とする 1024×1024 px の決定的なクロップに対して実行します。結果はこのクロップに関するものであり、論文の画像単位・品種単位の結果とは比較できません。

In [ ]:
from PIL import Image
import numpy as np

CROP = 1024
ys, xs = np.where(cls == 3)                       # parent-root pixels (class 3)
cy, cx = int(ys.mean()), int(xs.mean())           # deterministic crop center
y0 = int(np.clip(cy - CROP // 2, 0, cls.shape[0] - CROP))
x0 = int(np.clip(cx - CROP // 2, 0, cls.shape[1] - CROP))
crop = cls[y0:y0 + CROP, x0:x0 + CROP]
print(f"crop: rows {y0}:{y0+CROP}, cols {x0}:{x0+CROP} of {cls.shape}; "
      f"class counts: {dict(zip(*[a.tolist() for a in np.unique(crop, return_counts=True)]))}")
Image.fromarray(crop).save("/content/dirtmu/sample/TAKFA3-n3-1_Classes_crop1024.tiff")

vals_c, counts_c = np.unique(crop, return_counts=True)
assert {1, 2, 3}.issubset(set(vals_c.tolist())), "crop must contain all three classes"
rgb_crop = np.full(crop.shape + (3,), 255, dtype=np.uint8)
rgb_crop[crop == 1] = (214, 39, 40)
rgb_crop[crop == 3] = (44, 160, 44)
fig, ax = plt.subplots(figsize=(6, 6), dpi=110)
ax.imshow(rgb_crop, interpolation="nearest")
ax.set_title(f"Executed input: {CROP}x{CROP} crop of TAKFA3-n3-1_Classes.tiff\n(rows {y0}:{y0+CROP}, cols {x0}:{x0+CROP}; red = root hair, green = parent root)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

### 8. Run the author's pipeline (README example command, on the crop)

Execute the author's command form from the DIRT/mu README ("Running the sample file"), with all scientific parameters unchanged: `--id_root 3 --id_roothair 1 --id_background 2 --pixel_size 1.72`, author-default optimization settings (`--measure total_curvature`, `--cost_type rms`, `--n_levels 1000`, weights 1/1/1). The only change versus the README example is the input extent (the deterministic crop from §7) and the output directory `/content/dirtmu/out/` (keeps inputs and outputs separate; no computational effect). The `-p` flag additionally saves the per-hair/outlier tables, result PNGs, the classified image, and a metadata CSV. Per-stage elapsed times are printed by the pipeline itself; the simulated-annealing stage is the slowest.

**日本語:** 著者 README のサンプル実行コマンドを、入力を §7 の決定的クロップに、出力先を `/content/dirtmu/out/` に変えて実行します（それ以外のパラメータは著者既定値のまま）。`-p` により結果テーブル・PNG・メタデータも保存されます。

In [ ]:
!mkdir -p /content/dirtmu/out
!/content/envs/dirtmu/bin/python3 /content/dirtmu/hairyroots.py -i /content/dirtmu/sample/TAKFA3-n3-1_Classes_crop1024.tiff -o /content/dirtmu/out --id_root 3 --id_roothair 1 --id_background 2 --pixel_size 1.72 -p

### 9. Results — summary table

Read the summary CSV written by the author pipeline: extracted/inlier hair counts, total and mean lengths (µm, using the 1.72 µm/px calibration), density metrics, completeness ratio of the simulated-annealing solution, and the per-stage timings measured on this runtime.

**日本語:** パイプラインが出力したサマリー CSV（抽出・妥当な根毛数、合計・平均長さ（µm）、密度指標、解の完全性、各段階の実測時間）を表示します。

In [ ]:
import pandas as pd

RUN = "/content/dirtmu/out/TAKFA3-n3-1_Classes_crop1024"
summary = pd.read_csv(RUN + "_summary.csv")
assert len(summary) == 1, "expected a single-row summary"
display(summary.T.rename(columns={0: "value"}))

### 10. Results — per-hair measurements and length distribution

Show the per-root-hair table (length in µm, curvature and diameter statistics from the author's spline-curve fits, plus the root class/position assignment computed by `rh_density.computeDensity`), and plot the root hair length distribution. The histogram is a **notebook-generated visualization of the pipeline's own output**, not a figure from the paper.

**日本語:** 個々の根毛の計測テーブル（長さ・曲率・直径の統計量、根のクラス/位置）を表示し、根毛長の分布をプロットします。ヒストグラムはパイプライン出力の**ノートブック側での追加可視化**であり、論文の図ではありません。

In [ ]:
hairs = pd.read_csv(RUN + "_roothairs.csv")
print(f"measured root hairs: {len(hairs)} (inliers); columns: {list(hairs.columns)}")
display(hairs.head(10))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), dpi=110)
axes[0].hist(hairs["length_mu"], bins=40, color="#4c72b0")
axes[0].set_xlabel("root hair length (µm)"); axes[0].set_ylabel("count")
axes[0].set_title(f"Root hair length — notebook-generated\n(author sample crop 1024px, {len(hairs)} inlier hairs)")
axes[1].hist(hairs["diameter_mu_mean"], bins=40, color="#dd8452")
axes[1].set_xlabel("mean diameter (µm)"); axes[1].set_ylabel("count")
axes[1].set_title("Mean diameter — notebook-generated")
plt.tight_layout(); plt.show()
hairs.to_csv("/content/dirtmu/out/roothairs_export.csv", index=False)
print("CSV export available at /content/dirtmu/out/roothairs_export.csv (download from the file browser)")

### 11. Input vs. prediction overlay

Left: the input classified image (same rendering as the preview). Right: the author pipeline's own result figure (`*_roothairs.png`), which overlays the reconstructed individual hair curves on the input. Each curve is one measured root hair; hair identity colors are arbitrary. This is the paper's core claim in action — intersecting root-hair pixels are resolved into individual hairs rather than measured as blobs.

**日本語:** 左が入力の分類画像、右が著者パイプラインの結果図（`*_roothairs.png`、入力上に再構成された個々の根毛曲線を重畳）。曲線の色は識別用で意味はありません。交差する根毛ピクセルが個体として分離・計測されるのが本手法の核です。

In [ ]:
import matplotlib.image as mpimg

pred = mpimg.imread(RUN + "_roothairs.png")
fig, axes = plt.subplots(1, 2, figsize=(12, 6.2), dpi=110)
axes[0].imshow(rgb, interpolation="nearest")
axes[0].set_title("Input: 1024px crop of classified sample\n(red = root hair, white = background, green = parent root)")
axes[1].imshow(pred, interpolation="nearest")
axes[1].set_title(f"DIRT/mu predicted root hairs (author output roothairs.png)\n{len(hairs)} measured hairs, arbitrary identity colors")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### 12. Root hair density map

The author pipeline also computes root hair density along the parent root (`rh_density.computeDensity` / `plotDensity`): hairs are assigned to root edge classes and positions, and a rolling-sum density is reported. Display the author's density figure. Length/density values here are for this single sample only — the paper's cultivar-level statistics are out of scope.

**日本語:** 著者パイプラインによる根毛密度図（`*_density.png`）を表示します。ここでの値はこの 1 サンプルのみのものです。

In [ ]:
density_img = mpimg.imread(RUN + "_density.png")
fig, ax = plt.subplots(figsize=(10, 4), dpi=110)
ax.imshow(density_img, interpolation="nearest")
ax.set_title("DIRT/mu root hair density (author output density.png)")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Pipeline output files:")
import glob
for p in sorted(glob.glob("/content/dirtmu/out/*")):
    print(" ", p)

## Interpretation, limitations, and validation record

**What was executed:** the author's own DIRT/mu 1.1 pipeline (commit `d54aac0c38e194a84363496d8f17f9a0016d3bdc`) on the repository's bundled classified sample image, with the README's documented parameters. Measured outputs: per-hair length/curvature/diameter table, summary metrics, density figure, and result overlays — all displayed above from the actual run.

**Plausibility, not equivalence:** with 1.72 µm/px calibration the measured lengths are of the scale reported qualitatively for maize root hairs; no reference outputs ship with the repository and the preprint full text is inaccessible, so no quantitative comparison against published numbers is possible. A single-sample run does not verify the paper's dataset-level accuracy (length R² 0.72–0.85, density R² 0.38–0.66 vs. manual data, per the abstract).

**Runtime adaptation (measured):** the author-default full-image run (3072×4080 px) was executed here up to the merging stage: ~2 min preprocessing, 896,456 enumerated candidates (141,775 kept), and a measured merging rate of ~130–160 indices/minute projecting >11 additional hours — beyond a Colab session. The executed demonstration therefore uses a deterministic 1024×1024 px crop (centered on the parent-root centroid) with the pipeline and parameters otherwise unchanged. All results are for that crop.

**Known limitations:**
- The simulated annealing stage uses unseeded randomness (`random`/numpy); re-running may shift individual hair assignments slightly. Reported metric magnitudes are stable at this scale.
- The preprint body (ESSOAr) returned HTTP 403 on HTML/PDF/JATS routes; Europe PMC was unavailable (503); OpenAlex has no cached copy. Only the Crossref-hosted abstract is verified. Attribution of the repository to this preprint rests on the software name, first author, creation date (one day before preprint posting), and the later 2024 version's citation of the same repository.
- Out of scope (see header): ilastik classification from raw images, batch/cultivar analysis, and manual-validation statistics.

**References:** DIRT/mu 1.1 code, BSD-3-Clause — https://github.com/Computational-Plant-Science/DIRTmu (commit `d54aac0c38e194a84363496d8f17f9a0016d3bdc`); paper DOI 10.1002/essoar.10508833.1; sample image and all parameters from the repository README at that commit.

**日本語:** 実行したのは著者パイプラインそのもの（README 記載コマンド・既定パラメータ）であり、出力はすべてこの実行の実測値です。リポジトリには参照出力が同梱されず、論文全文も入手不可のため、公開数値との定量比較はできません（1 サンプル実行はデータセットレベルの精度検証ではありません）。焼きなましはシードなしの乱数を用いるため、再実行で個別の根毛の帰属が多少変わり得ます。実行範囲は親根重心中心の 1024×1024 px クロップです（全画像の既定実行は実測 12 時間超のため Colab セッションに収まらず、クロップで代行。アルゴリズム・パラメータは不変）。範囲外：ilastik 分類、バッチ処理、手動検証統計。